# Gen Set Enrichment Analysis with GSEApy - A549 LOY vs ROY

### Import

In [ ]:
import os
import pickle

import pandas as pd
import seaborn as sns
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import plotly.express as px
import scipy.cluster.hierarchy as sch

%matplotlib inline 

from gseapy import barplot, dotplot
from gseapy import gseaplot

In [ ]:
# Set directories
data_dir = '/data'

In [18]:
# Load combined tpm and count matrix with DESeq2 results (LOY vs ROY)
df = pd.read_csv(os.path.join(data_dir,'Table3.csv'), sep=',', header=0, index_col=None) # Table 3 is provided as a supplementary table

In [ ]:
# Load gene sets
MSigDB_Hallmark_2020 = pd.read_csv(os.path.join(data_dir,'MSigDB_Hallmark_2020.csv'), delimiter=';', header=None) #can be downloaded from: https://www.gsea-msigdb.org/gsea/msigdb/human/collections.jsp#H

In [ ]:
# Convert to dictionary format
hall_dict = {}
for _, row in MSigDB_Hallmark_2020.iterrows():
    # Skip rows where the first column is NaN or empty
    if pd.notna(row[0]) and row[0].strip():
        hall_dict[row[0]] = list(row[2:].dropna().astype(str))

### Prerank API

In [19]:
# prepare ranked gene list
# sort by log2FoldChange
sorted_gene_list_a549 = df.sort_values(by='log2FoldChange', ascending=False)

# Subset the DataFrame to include only the gene name column
glist_a549_rnk = sorted_gene_list_a549[['gene_name', 'log2FoldChange']]

# Set 'gene_name' as the index
glist_a549_rnk.set_index('gene_name', inplace=True)

In [ ]:
pre_res_a549_hall = gp.prerank(rnk=glist_a549_rnk, 
                     gene_sets=hall_dict, 
                     threads=4,
                     min_size=5, 
                     max_size=1000, 
                     permutation_num=1000, 
                     outdir=output_dir, 
                     seed=6,
                     verbose=True, 
                    )

### Dotplot - Fig 2D

In [ ]:
# Dotplot (Fig 2D)
plt.style.use("default")

ax = dotplot(pre_res_a549_hall.res2d,
             column="FDR q-val",
             cmap=plt.cm.viridis,
             size=10, 
             top_term = 10,
             figsize=(3,7), show_ring=True, cutoff=0.25)

ax.set_title('A549 LOY vs ROY', fontsize=12)
ax.set_xlabel('NES', fontsize=12)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)

fig = ax.figure
fig.savefig(
    "A549_LOY_vs_ROY_dotplot.pdf",
    format="pdf",
    bbox_inches="tight"
)

### Heatmap with lead EMT genes - Fig 2E

In [ ]:
# load EMT lead gene list (results from prerank analysis)
emt_lists = pd.read_csv(os.path.join(data_dir, 'EMT_genes_GSEA_A549.csv'), header=None, sep = ",", engine='python', names=range(201))

# extract prerank_GSEA row
prerank_emt = emt_lists.iloc[1] 

# convert to list
prerank_emt_list = prerank_emt.tolist()

# drop None and name of the row
prerank_emt_list_clean = [x for x in prerank_emt_list if x is not None]
prerank_emt_list_fin = prerank_emt_list_clean[1:]

In [ ]:
# identify tpm columns in combined matrix
tpm_cols = [c for c in df.columns if c.endswith('_tpm')]
tpm_cols

In [ ]:
# subset df for EMT genes (lead genes from prerank results)
df_pre_emt = df[df['gene_name'].isin(prerank_emt_list_fin)].copy()

# create a matrix for heatmap (genes as rows and samples as columns)
heatmap_pre_emt = df_pre_emt.set_index('gene_name')[tpm_cols]

# log2 transform
heatmap_pre_emt = np.log2(heatmap_pre_emt + 1)  

# z-score normalization
heatmap_pre_emt_z = heatmap_pre_emt.sub(heatmap_pre_emt.mean(axis=1), axis=0)
heatmap_pre_emt_z = heatmap_pre_emt_z.div(heatmap_pre_emt.std(axis=1), axis=0)

heatmap_pre_emt_z = heatmap_pre_emt_z.sort_index(key=lambda x: x.str.lower())

In [ ]:
# heatmap without clustering (Fig 2E)

g = sns.clustermap(
    heatmap_pre_emt_z,
    cmap='seismic',
    row_cluster=False,  
    col_cluster=False,  
    yticklabels=heatmap_pre_emt_z.index.tolist(),
    xticklabels=heatmap_pre_emt_z.columns.tolist(),
    figsize=(12, 18)
)

for label in g.ax_heatmap.get_yticklabels():
    label.set_fontsize(12)
for label in g.ax_heatmap.get_xticklabels():
    label.set_fontsize(10)

plt.show()